# Phase 1: Dataset Audit — Interactive Validation Notebook
## Rice Plant Disease Classification (CC + MCLBP + Linear SVM)

This notebook provides **visual and interactive validation** of the automated
Phase 1 audit results produced by `audit_engine.py`. It loads the generated
CSV deliverables, plots class distributions, shows sample images from each
category, visualises duplicate/quarantine statistics, and displays
near-duplicate candidate pairs for human review.

**Prerequisites**: Run `python audit/run_audit.py` first to generate all CSV
outputs in the `audit/` directory.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from collections import Counter
from PIL import Image

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
})

# Paths — adjust if running from a different working directory
PROJECT_ROOT = Path('.').resolve()
AUDIT_DIR = PROJECT_ROOT / 'audit'
RAW_DIR = PROJECT_ROOT / 'Data' / 'Raw'

print(f'Project root : {PROJECT_ROOT}')
print(f'Audit dir    : {AUDIT_DIR}')
print(f'Raw data dir : {RAW_DIR}')

---
## 1. Load Audit Outputs

In [ ]:
manifest = pd.read_csv(AUDIT_DIR / 'dataset_manifest_initial.csv')
class_dist = pd.read_csv(AUDIT_DIR / 'class_distribution.csv')
exact_dups = pd.read_csv(AUDIT_DIR / 'exact_duplicates.csv')
exact_groups = pd.read_csv(AUDIT_DIR / 'exact_duplicate_groups.csv')
near_dups = pd.read_csv(AUDIT_DIR / 'near_duplicate_candidates.csv')

print(f'Manifest rows           : {len(manifest):,}')
print(f'Class distribution rows  : {len(class_dist)}')
print(f'Exact duplicate records  : {len(exact_dups):,}')
print(f'Exact duplicate groups   : {len(exact_groups):,}')
print(f'Near-duplicate pairs     : {len(near_dups):,}')
print()
print('Manifest columns:')
print(list(manifest.columns))

---
## 2. Raw Image Count per Dataset × Class

In [ ]:
pivot_raw = manifest.pivot_table(
    index='canonical_label', columns='dataset_id',
    values='sha256', aggfunc='count', fill_value=0
)

fig, ax = plt.subplots(figsize=(9, 5))
pivot_raw.plot(kind='bar', ax=ax, edgecolor='black', linewidth=0.5)
ax.set_title('Raw Image Count per Canonical Class × Dataset')
ax.set_xlabel('Canonical Label')
ax.set_ylabel('Image Count')
ax.legend(title='Dataset')
for container in ax.containers:
    ax.bar_label(container, fmt='%d', fontsize=7, padding=2)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(str(AUDIT_DIR / 'plot_raw_class_distribution.png'), dpi=150)
plt.show()

---
## 3. Recommended (Deduplicated, Conflict-Free) Distribution

In [ ]:
recommended = manifest[manifest['is_canonical_recommendation'] == True]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart
rec_counts = recommended['canonical_label'].value_counts().sort_index()
colors = ['#e74c3c', '#3498db', '#e67e22', '#2ecc71', '#9b59b6', '#1abc9c']
axes[0].pie(
    rec_counts.values, labels=rec_counts.index,
    autopct='%1.1f%%', startangle=140,
    colors=colors[:len(rec_counts)],
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
axes[0].set_title(f'Recommended Clean Images\n(N = {len(recommended):,})')

# Bar chart
rec_counts.plot(kind='bar', ax=axes[1], color=colors[:len(rec_counts)],
                edgecolor='black', linewidth=0.5)
axes[1].set_title('Recommended Images per Class')
axes[1].set_xlabel('Canonical Label')
axes[1].set_ylabel('Count')
for container in axes[1].containers:
    axes[1].bar_label(container, fmt='%d', fontsize=9, padding=3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(str(AUDIT_DIR / 'plot_recommended_distribution.png'), dpi=150)
plt.show()

---
## 4. Quarantine & Duplicate Breakdown

In [ ]:
status_counts = manifest['quarantine_status'].value_counts()
print('Quarantine Status Distribution:')
print(status_counts)
print()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Quarantine breakdown
q_colors = {'CLEAN': '#27ae60', 'REDUNDANT': '#f39c12', 'QUARANTINED': '#e74c3c'}
status_counts.plot(
    kind='bar', ax=axes[0],
    color=[q_colors.get(s, '#95a5a6') for s in status_counts.index],
    edgecolor='black', linewidth=0.5
)
axes[0].set_title('Image Status Distribution')
axes[0].set_ylabel('Count')
for container in axes[0].containers:
    axes[0].bar_label(container, fmt='%d', fontsize=9)

# Exact duplicate group sizes
if not exact_groups.empty and 'group_size' in exact_groups.columns:
    group_sizes = exact_groups['group_size']
    axes[1].hist(group_sizes, bins=range(1, int(group_sizes.max()) + 2),
                 color='#3498db', edgecolor='black', linewidth=0.5, alpha=0.85)
    axes[1].set_title(f'Exact Duplicate Group Sizes\n({len(exact_groups):,} groups)')
    axes[1].set_xlabel('Group Size (number of copies)')
    axes[1].set_ylabel('Number of Groups')
else:
    axes[1].text(0.5, 0.5, 'No group_size column found', ha='center', va='center',
                 transform=axes[1].transAxes)

plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(str(AUDIT_DIR / 'plot_quarantine_breakdown.png'), dpi=150)
plt.show()

---
## 5. Image Dimension Analysis

In [ ]:
print('Image dimension statistics:')
print(manifest[['image_width', 'image_height', 'file_size_bytes']].describe())
print()

dim_counts = manifest.groupby(['image_width', 'image_height']).size().sort_values(ascending=False)
print('Top 10 image dimensions (W × H):')
for (w, h), count in dim_counts.head(10).items():
    print(f'  {w} × {h}: {count:,} images ({100*count/len(manifest):.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Scatter: width vs height, coloured by dataset
for ds_id, group in manifest.groupby('dataset_id'):
    axes[0].scatter(group['image_width'], group['image_height'],
                    alpha=0.3, s=8, label=ds_id)
axes[0].set_xlabel('Width (px)')
axes[0].set_ylabel('Height (px)')
axes[0].set_title('Image Dimensions by Dataset')
axes[0].legend(markerscale=3)

# Histogram of file sizes
axes[1].hist(manifest['file_size_bytes'] / 1024, bins=50,
             color='#8e44ad', edgecolor='black', linewidth=0.3, alpha=0.8)
axes[1].set_xlabel('File Size (KB)')
axes[1].set_ylabel('Count')
axes[1].set_title('File Size Distribution')

plt.tight_layout()
plt.savefig(str(AUDIT_DIR / 'plot_dimensions_filesizes.png'), dpi=150)
plt.show()

---
## 6. Sample Image Grid per Canonical Class

Displays 4 random **recommended** (clean, deduplicated) images from each class.

In [ ]:
classes = sorted(recommended['canonical_label'].unique())
n_samples = 4

fig, axes = plt.subplots(len(classes), n_samples, figsize=(14, 3 * len(classes)))
if len(classes) == 1:
    axes = [axes]

for row_idx, cls in enumerate(classes):
    cls_df = recommended[recommended['canonical_label'] == cls]
    samples = cls_df.sample(n=min(n_samples, len(cls_df)), random_state=42)
    for col_idx in range(n_samples):
        ax = axes[row_idx][col_idx]
        if col_idx < len(samples):
            img_path = samples.iloc[col_idx]['image_path']
            try:
                img = Image.open(img_path)
                ax.imshow(img)
                w, h = samples.iloc[col_idx]['image_width'], samples.iloc[col_idx]['image_height']
                ax.set_title(f'{cls} ({w}×{h})', fontsize=9)
            except Exception:
                ax.text(0.5, 0.5, 'Load Error', ha='center', va='center',
                        transform=ax.transAxes, color='red')
        ax.axis('off')
    axes[row_idx][0].set_ylabel(cls, fontsize=12, fontweight='bold',
                                 rotation=0, labelpad=40, va='center')

plt.suptitle('Sample Images per Canonical Disease Class (Recommended Subset)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(str(AUDIT_DIR / 'plot_sample_images_grid.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Critical Anomaly: DS2 Leafsmut ≡ DS3 Blast

Visual confirmation that `Data - 2/Leafsmut` images are **byte-identical**
to `Data - 3/Blast` images.

In [ ]:
quarantined = manifest[manifest['quarantine_status'] == 'QUARANTINED']
print(f'Total quarantined images: {len(quarantined):,}')
print(f'Quarantined by label:')
print(quarantined['canonical_label'].value_counts())
print()

# Show a few matching pairs
leafsmut_q = quarantined[quarantined['original_class'] == 'Leafsmut']
blast_q = quarantined[quarantined['original_class'] == 'Blast']

if not leafsmut_q.empty and not blast_q.empty:
    # Find matching SHA-256 pairs
    common_hashes = set(leafsmut_q['sha256']) & set(blast_q['sha256'])
    sample_hashes = list(common_hashes)[:3]

    fig, axes = plt.subplots(len(sample_hashes), 2, figsize=(8, 4 * len(sample_hashes)))
    if len(sample_hashes) == 1:
        axes = [axes]

    for i, sha in enumerate(sample_hashes):
        ls_row = leafsmut_q[leafsmut_q['sha256'] == sha].iloc[0]
        bl_row = blast_q[blast_q['sha256'] == sha].iloc[0]

        for j, (row, label) in enumerate([(ls_row, 'DS2/Leafsmut'), (bl_row, 'DS3/Blast')]):
            try:
                img = Image.open(row['image_path'])
                axes[i][j].imshow(img)
            except Exception:
                axes[i][j].text(0.5, 0.5, 'Load Error', ha='center',
                                va='center', transform=axes[i][j].transAxes)
            axes[i][j].set_title(f'{label}\n{row["file_name"]}', fontsize=8)
            axes[i][j].axis('off')

    plt.suptitle('Quarantined: Identical Images with Conflicting Labels\n'
                 '(DS2 "Leafsmut" ≡ DS3 "Blast", same SHA-256)',
                 fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(str(AUDIT_DIR / 'plot_leafsmut_blast_conflict.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No quarantined Leafsmut/Blast pairs found to display.')

---
## 8. Near-Duplicate Pair Inspection

Displays the top near-duplicate candidate pairs with **conflicting labels**
(highest priority for manual review).

In [ ]:
# Check for near-dup label conflicts
if 'same_canonical_label' in near_dups.columns:
    conflicts = near_dups[near_dups['same_canonical_label'] == False].copy()
    print(f'Near-duplicate pairs with conflicting labels: {len(conflicts)}')

    if len(conflicts) > 0:
        # Show first 4 conflict pairs visually
        n_show = min(4, len(conflicts))
        fig, axes = plt.subplots(n_show, 2, figsize=(8, 4 * n_show))
        if n_show == 1:
            axes = [axes]

        for i in range(n_show):
            row = conflicts.iloc[i]
            for j, suffix in enumerate(['_a', '_b']):
                path_col = f'image_path{suffix}'
                label_col = f'canonical_label{suffix}'
                fname_col = f'file_name{suffix}'
                if path_col in row.index:
                    try:
                        img = Image.open(row[path_col])
                        axes[i][j].imshow(img)
                    except Exception:
                        axes[i][j].text(0.5, 0.5, 'Load Error', ha='center',
                                        va='center', transform=axes[i][j].transAxes)
                    lbl = row.get(label_col, '?')
                    fn = row.get(fname_col, '?')
                    axes[i][j].set_title(f'{lbl}: {fn}', fontsize=8)
                axes[i][j].axis('off')

        plt.suptitle('Near-Duplicate Pairs with Label Conflicts\n'
                     '(Require manual review)', fontsize=12, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.savefig(str(AUDIT_DIR / 'plot_near_dup_conflicts.png'), dpi=150, bbox_inches='tight')
        plt.show()
    else:
        print('No near-duplicate label conflicts found.')
else:
    print('Near-duplicate CSV does not contain same_canonical_label column.')
    print('Columns:', list(near_dups.columns))

---
## 9. DS1 Train/Test Leakage Verification

In [ ]:
ds1 = manifest[manifest['dataset_id'] == 'DS1']
ds1_train = ds1[ds1['hierarchy_context'].str.contains('Full_Train', na=False)]
ds1_test = ds1[ds1['hierarchy_context'].str.contains('Test_Data', na=False)]

train_hashes = set(ds1_train['sha256'])
test_hashes = set(ds1_test['sha256'])
overlap = train_hashes & test_hashes

print(f'DS1 Train images: {len(ds1_train):,} ({len(train_hashes):,} unique SHA-256)')
print(f'DS1 Test images:  {len(ds1_test):,} ({len(test_hashes):,} unique SHA-256)')
print(f'SHA-256 overlap (train ∩ test): {len(overlap)} hash(es)')

if overlap:
    print('\nLeaked images:')
    for h in overlap:
        train_match = ds1_train[ds1_train['sha256'] == h][['relative_path', 'canonical_label']]
        test_match = ds1_test[ds1_test['sha256'] == h][['relative_path', 'canonical_label']]
        print(f'  SHA-256: {h[:16]}...')
        print(f'    Train: {train_match.iloc[0]["relative_path"]}')
        print(f'    Test:  {test_match.iloc[0]["relative_path"]}')
else:
    print('No train/test leakage detected.')

---
## 10. DS2 ⊆ DS3 Subset Verification

In [ ]:
ds2 = manifest[manifest['dataset_id'] == 'DS2']
ds3 = manifest[manifest['dataset_id'] == 'DS3']

ds2_hashes = set(ds2['sha256'])
ds3_hashes = set(ds3['sha256'])

ds2_in_ds3 = ds2_hashes & ds3_hashes
ds2_only = ds2_hashes - ds3_hashes
ds3_only = ds3_hashes - ds2_hashes

print(f'DS2 unique hashes: {len(ds2_hashes):,}')
print(f'DS3 unique hashes: {len(ds3_hashes):,}')
print(f'DS2 ∩ DS3:         {len(ds2_in_ds3):,} ({100*len(ds2_in_ds3)/len(ds2_hashes):.1f}% of DS2)')
print(f'DS2 only:          {len(ds2_only):,}')
print(f'DS3 only:          {len(ds3_only):,}')
print()

if len(ds2_in_ds3) == len(ds2_hashes):
    print('CONFIRMED: DS2 is a complete subset of DS3.')
    print('DS2 can be safely excluded from the training pool without losing any unique content.')
else:
    print(f'WARNING: {len(ds2_only)} unique images exist only in DS2.')

---
## 11. Summary Statistics Table

In [ ]:
print('='*70)
print('PHASE 1 AUDIT — FINAL SUMMARY')
print('='*70)

summary_data = {
    'Total raw images': len(manifest),
    'Unique SHA-256 hashes': manifest['sha256'].nunique(),
    'Exact duplicate groups': len(exact_groups),
    'Quarantined (label conflict)': len(manifest[manifest['quarantine_status']=='QUARANTINED']),
    'Redundant (exact copies)': len(manifest[manifest['quarantine_status']=='REDUNDANT']),
    'Recommended clean images': len(recommended),
    'Near-duplicate candidate pairs': len(near_dups),
    'Unreadable / corrupted': 0,
    'Unmapped labels': 0,
}

for k, v in summary_data.items():
    print(f'  {k:40s}: {v:>8,}')

print()
print('Recommended class distribution:')
for cls in sorted(recommended['canonical_label'].unique()):
    count = len(recommended[recommended['canonical_label'] == cls])
    print(f'  {cls:5s}: {count:>6,}')

print('='*70)

---
## 12. Phase 2 Readiness Checklist

| # | Item | Status |
|---|------|--------|
| 1 | All images scanned and verified | ✅ |
| 2 | SHA-256 exact duplicates identified | ✅ |
| 3 | Perceptual near-duplicates catalogued | ✅ |
| 4 | Label mapping verified | ✅ |
| 5 | DS2 Leafsmut/DS3 Blast anomaly documented & quarantined | ✅ |
| 6 | DS1 train/test leakage documented | ✅ |
| 7 | DS2 ⊆ DS3 redundancy confirmed | ✅ |
| 8 | Recommended clean subset identified (6,632 images) | ✅ |
| 9 | All CSV/MD/JSON deliverables generated | ✅ |

**Next**: Resolve quarantined Leafsmut/Blast labels → construct leakage-safe
train/val/test splits → proceed to Phase 2 feature extraction (CC + MCLBP).